# RankUp - Energy Efficiency UCI

Notebook ini berisi implementasi sederhana RankUp pada dataset **Energy Efficiency UCI**.

Urutan eksperimen:

1. **Supervised**: hanya memakai data berlabel (`N_LABEL`).
2. **RankUp minimal**: memakai data berlabel + data unlabeled melalui pseudo-label ranking.
3. **Fully-Supervised**: memakai semua label train sebagai pembanding atas.

Target regresi yang dipakai adalah `Y1` atau **Heating Load**.

## 1. Import dan Konfigurasi

Cell ini menyiapkan library, hyperparameter, path dataset, seed, dan device training.

Penjelasan syntax penting:

- `import ...`: memanggil library yang akan dipakai di notebook.
- `torch`: library deep learning untuk membuat model dan training loop.
- `torch.nn as nn`: modul PyTorch untuk layer model dan loss function.
- `matplotlib.pyplot`: library dasar untuk membuat figure dan mengatur plot.
- `seaborn`: library visualisasi berbasis matplotlib yang lebih rapi untuk metric, distribusi, dan scatter plot.
- `DataLoader`: membuat mini-batch dari dataset.
- `TensorDataset`: membungkus tensor input dan target menjadi dataset PyTorch sederhana.
- `train_test_split`: membagi data menjadi train dan test.
- `StandardScaler`: menormalkan fitur dan target agar training lebih stabil.
- `torch.device(...)`: memilih GPU (`cuda`) jika tersedia, kalau tidak memakai CPU.
- `N_LABEL`: jumlah data train berlabel yang dipakai.
- `W_RANK`: bobot loss ranking classifier.
- `W_UNLAB`: bobot FixMatch loss untuk data unlabeled.
- `P_CUTOFF`: threshold confidence; pseudo-label ranking dipakai hanya jika confidence weak ARC cukup tinggi.
- `NOISE_WEAK` dan `NOISE_STRONG`: noise kecil dan besar untuk simulasi augmentasi pada data tabular.

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn

from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.manifold import TSNE
from scipy.stats import spearmanr

SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Konfigurasi utama eksperimen.
N_LABEL = 80
BATCH_SIZE = 32
EPOCHS_BASE = 120
EPOCHS_RANKUP = 120
EPOCHS_FULL = 120
LR = 1e-3
W_RANK = 0.2
W_UNLAB = 0.2
P_CUTOFF = 0.95
NOISE_WEAK = 0.03
NOISE_STRONG = 0.10

# Path dataset: lokal Kaggle dipakai jika tersedia, URL UCI dipakai jika internet aktif.
UCI_URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/00242/ENB2012_data.xlsx"
LOCAL_DATA_PATH = "/kaggle/input/energy-efficiency/ENB2012_data.xlsx"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (8, 4)

print("Device:", DEVICE)

## 2. Load Dataset

Cell ini membaca dataset Energy Efficiency.

Alur kode:

1. Membuat daftar nama kolom dataset.
2. Mengecek apakah file lokal Kaggle tersedia melalui `os.path.exists()`.
3. Jika file lokal ada, dataset dibaca dari `LOCAL_DATA_PATH`.
4. Jika file lokal tidak ada, dataset dibaca langsung dari URL UCI.
5. Nama kolom dirapikan supaya mudah dipanggil.

Penjelasan syntax penting:

- `def load_energy_efficiency():` membuat fungsi agar proses load dataset bisa dipanggil ulang.
- `pd.read_excel(...)` membaca file Excel menjadi `DataFrame`.
- `data.columns = columns` mengganti nama kolom bawaan dataset.
- `dropna()` menghapus baris kosong jika ada.
- `reset_index(drop=True)` merapikan ulang index setelah data dibersihkan.
- `data.head()` menampilkan lima baris pertama untuk mengecek isi dataset.

In [ ]:
def load_energy_efficiency():
    columns = ["X1", "X2", "X3", "X4", "X5", "X6", "X7", "X8", "Y1", "Y2"]

    if os.path.exists(LOCAL_DATA_PATH):
        data = pd.read_excel(LOCAL_DATA_PATH)
    else:
        data = pd.read_excel(UCI_URL)

    data.columns = columns
    return data.dropna().reset_index(drop=True)


data = load_energy_efficiency()
print(data.shape)
data.head()

## 3. Penjelasan Variabel Dataset

Dataset **Energy Efficiency** berisi karakteristik desain bangunan dan dua target beban energi. Setiap baris merepresentasikan satu konfigurasi bangunan.

| Variable | Role | Type | Description | Missing Values |
|---|---|---|---|---|
| `X1` | Feature | Continuous | Relative Compactness | No |
| `X2` | Feature | Continuous | Surface Area | No |
| `X3` | Feature | Continuous | Wall Area | No |
| `X4` | Feature | Continuous | Roof Area | No |
| `X5` | Feature | Continuous | Overall Height | No |
| `X6` | Feature | Integer | Orientation | No |
| `X7` | Feature | Continuous | Glazing Area | No |
| `X8` | Feature | Integer | Glazing Area Distribution | No |
| `Y1` | Target | Continuous | Heating Load | No |
| `Y2` | Target | Continuous | Cooling Load | No |

Penjelasan ringkas:

- `X1` sampai `X8` adalah fitur input yang menjelaskan bentuk dan komponen bangunan.
- `Y1` adalah target **Heating Load**, yaitu kebutuhan energi untuk pemanasan.
- `Y2` adalah target **Cooling Load**, yaitu kebutuhan energi untuk pendinginan.
- Notebook ini hanya memakai `Y1` sebagai target agar eksperimen tetap sederhana dan fokus pada konsep RankUp.
- Semua kolom tidak memiliki missing value, sehingga tidak diperlukan imputasi data.

Catatan tentang tipe data:

- Kolom bertipe `Continuous` berisi nilai numerik kontinu.
- Kolom bertipe `Integer` berisi kategori yang direpresentasikan sebagai angka, seperti `Orientation` dan `Glazing Area Distribution`.
- Pada notebook ini, semua fitur tetap dipakai sebagai input numerik sederhana agar preprocessing tidak terlalu rumit.

## 4. Split Data

Split dilakukan dalam dua tahap.

Pertama, dataset dipisah menjadi train dan test:

```text
data total -> train set + test set
```

Kedua, train set dipisah lagi menjadi labeled dan unlabeled:

```text
train set -> labeled train + unlabeled train
```

Contoh jika ada 100 data dan test split 20%:

```text
100 data total
-> 80 train
-> 20 test
```

Jika `N_LABEL = 50`, maka:

```text
80 train
-> 50 labeled train
-> 30 unlabeled train

20 test
-> hanya untuk evaluasi
```

Jadi `N_LABEL` dihitung dari train set, bukan dari seluruh dataset. Test set tidak dipakai untuk training.

Notebook juga menyimpan `x_train` dan `y_train` untuk eksperimen Fully-Supervised. Bagian ini hanya dipakai sebagai pembanding atas, bukan sebagai setting semi-supervised.

In [ ]:
x_all = data[["X1", "X2", "X3", "X4", "X5", "X6", "X7", "X8"]].values.astype("float32")
y_all = data["Y1"].values.astype("float32").reshape(-1, 1)

# Test set dipisahkan dulu dan tidak dipakai saat training.
x_train_raw, x_test_raw, y_train_raw, y_test_raw = train_test_split(
    x_all, y_all, test_size=0.2, random_state=SEED
)

# Dari train set, hanya N_LABEL sample yang labelnya dipakai oleh supervised dan RankUp.
x_lab_raw, x_unlab_raw, y_lab_raw, _ = train_test_split(
    x_train_raw, y_train_raw, train_size=N_LABEL, random_state=SEED
)

x_scaler = StandardScaler()
y_scaler = StandardScaler()

# Fitur semua train data boleh dipakai karena unlabeled data tetap punya fitur.
x_train = x_scaler.fit_transform(x_train_raw).astype("float32")
x_lab = x_scaler.transform(x_lab_raw).astype("float32")
x_unlab = x_scaler.transform(x_unlab_raw).astype("float32")
x_test = x_scaler.transform(x_test_raw).astype("float32")

# Target scaler fit dari labeled target agar supervised/RankUp tidak melihat label unlabeled.
y_lab = y_scaler.fit_transform(y_lab_raw).astype("float32")
y_train = y_scaler.transform(y_train_raw).astype("float32")
y_test = y_scaler.transform(y_test_raw).astype("float32")

print("train full:", x_train.shape)
print("labeled train:", x_lab.shape)
print("unlabeled train:", x_unlab.shape)
print("test:", x_test.shape)


## 5. DataLoader

Cell ini membuat empat loader:

- `lab_loader`: data labeled, berisi `x_lab` dan `y_lab`.
- `unlab_loader`: data unlabeled, hanya berisi `x_unlab`.
- `train_loader`: semua data train dengan label asli, khusus Fully-Supervised.
- `test_loader`: data test untuk evaluasi semua model.

Intinya:

```text
Supervised       -> lab_loader
RankUp           -> lab_loader + unlab_loader
Fully-Supervised -> train_loader
Evaluasi         -> test_loader
```

In [ ]:
x_lab_tensor = torch.from_numpy(x_lab).float()
y_lab_tensor = torch.from_numpy(y_lab).float()
x_unlab_tensor = torch.from_numpy(x_unlab).float()
x_train_tensor = torch.from_numpy(x_train).float()
y_train_tensor = torch.from_numpy(y_train).float()
x_test_tensor = torch.from_numpy(x_test).float()
y_test_tensor = torch.from_numpy(y_test).float()

lab_dataset = TensorDataset(x_lab_tensor, y_lab_tensor)
unlab_dataset = TensorDataset(x_unlab_tensor)
train_dataset = TensorDataset(x_train_tensor, y_train_tensor)
test_dataset = TensorDataset(x_test_tensor, y_test_tensor)

lab_loader = DataLoader(lab_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
unlab_loader = DataLoader(unlab_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False, drop_last=False)


## 6. Model

Cell ini mendefinisikan model MLP sederhana untuk regression dan RankUp.

Model memiliki tiga bagian:

1. `encoder`: mengubah fitur input menjadi fitur hidden.
2. `regressor`: memprediksi nilai target regresi.
3. `ranker`: classifier tambahan untuk belajar urutan/ranking, yaitu classifier 2 kelas untuk ranking pairwise.

Penjelasan syntax penting:

- `class RankUpMLP(nn.Module)`: membuat model PyTorch baru.
- `super().__init__()`: menjalankan inisialisasi bawaan `nn.Module`.
- `nn.Sequential(...)`: menyusun layer secara berurutan.
- `nn.Linear(a, b)`: layer fully-connected dari ukuran `a` ke ukuran `b`.
- `nn.ReLU()`: fungsi aktivasi agar model bisa belajar pola nonlinear.
- `forward(...)`: fungsi yang dipanggil saat model menerima input.
- `feat = self.encoder(x)`: mengambil representasi hidden dari input.
- `y_pred = self.regressor(feat)`: output utama untuk regresi.
- `rank_scores = self.ranker(z)`: output tambahan untuk ranking classifier.

Fungsi `pair_rank_logits` dan `pairwise_rank_target` adalah inti RankUp minimal:

- `pairwise_arc_logits(...)` membuat prediksi ranking untuk semua pasangan sample dalam batch.
- `pairwise_rank_target(...)` membuat label ranking dari target regresi.

In [ ]:
class RankUpMLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
        )
        self.regressor = nn.Linear(32, 1)
        self.ranker = nn.Linear(32, 2)

    def forward(self, features):
        z = self.encoder(features)
        y_pred = self.regressor(z)
        rank_scores = self.ranker(z)
        return y_pred, rank_scores


def pair_rank_logits(rank_scores):
    # Membuat skor ranking untuk semua pasangan sample dalam satu batch.
    return (rank_scores[:, None, :] - rank_scores[None, :, :]).reshape(-1, 2)


def pairwise_rank_target(targets):
    # Soft target mengikuti paper: lebih besar=1, lebih kecil=0, sama=0.5.
    target_diff = targets[:, None, :] - targets[None, :, :]
    rank_targets = (torch.sign(target_diff).reshape(-1) + 1) / 2
    return torch.stack([1 - rank_targets, rank_targets], dim=1)


## 7. Evaluasi

Cell ini menghitung performa model pada test set.

Metric yang dipakai:

- `MAE`: rata-rata kesalahan absolut. Makin kecil makin baik.
- `R2`: seberapa baik model menjelaskan variasi target. Makin besar makin baik.
- `Spearman`: seberapa baik model menjaga urutan/ranking target. Ini penting untuk RankUp.

Penjelasan syntax penting:

- `predict_test(model)`: mengambil prediksi dan target asli pada test set.
- `model.eval()`: mengubah model ke mode evaluasi.
- `with torch.no_grad()`: mematikan perhitungan gradient agar evaluasi lebih hemat memori.
- `pred.cpu().numpy()`: memindahkan hasil prediksi dari GPU ke CPU lalu menjadi NumPy array.
- `np.vstack(...)`: menggabungkan hasil prediksi dari banyak batch.
- `inverse_transform(...)`: mengembalikan skala target ke nilai asli sebelum standardisasi.
- `return {...}`: mengembalikan metric dalam bentuk dictionary.

In [ ]:
def predict_test(model):
    model.eval()
    pred_batches, target_batches = [], []

    with torch.no_grad():
        for x_test_batch, y_test_batch in test_loader:
            x_test_batch = x_test_batch.to(DEVICE)
            pred_batch, _ = model(x_test_batch)
            pred_batches.append(pred_batch.cpu().numpy())
            target_batches.append(y_test_batch.numpy())

    pred_scaled = np.vstack(pred_batches)
    target_scaled = np.vstack(target_batches)

    pred_real = y_scaler.inverse_transform(pred_scaled).reshape(-1)
    target_real = y_scaler.inverse_transform(target_scaled).reshape(-1)
    return pred_real, target_real


def evaluate(model):
    pred_real, target_real = predict_test(model)

    return {
        "MAE": mean_absolute_error(target_real, pred_real),
        "R2": r2_score(target_real, pred_real),
        "Spearman": spearmanr(target_real, pred_real).correlation,
    }


def print_metrics(name, metrics):
    print(name)
    for key, value in metrics.items():
        print(f"  {key}: {value:.4f}")


## 8. Supervised Baseline

Model pertama adalah regresi biasa dengan label sedikit.

Model ini hanya memakai:

```text
labeled features + labeled target
```

Data unlabeled tidak dipakai. Tujuannya menjadi baseline: seberapa baik model jika hanya dilatih dengan `N_LABEL` label.

In [ ]:
def train_supervised():
    model = RankUpMLP(input_dim=x_lab.shape[1]).to(DEVICE)
    loss_fn = nn.L1Loss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    history = []

    for epoch in range(EPOCHS_BASE):
        model.train()
        total_loss = 0

        for x_lab_batch, y_lab_batch in lab_loader:
            x_lab_batch = x_lab_batch.to(DEVICE)
            y_lab_batch = y_lab_batch.to(DEVICE)

            pred_batch, _ = model(x_lab_batch)
            loss = loss_fn(pred_batch, y_lab_batch)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        history.append(total_loss / len(lab_loader))

    return model, history


supervised_model, supervised_history = train_supervised()
supervised_metrics = evaluate(supervised_model)
print_metrics("Supervised baseline", supervised_metrics)


## 9. RankUp Minimal

RankUp minimal memakai dua jenis data saat training:

```text
labeled   : labeled features + labeled target
unlabeled : unlabeled features saja
```

Target utama tetap regresi `Y1`, tetapi model juga diberi tugas tambahan: belajar **ranking pairwise** lewat Auxiliary Ranking Classifier (ARC).

### Ide Inti

Regresi biasa bertanya:

```text
berapa nilai Y1 untuk sample ini?
```

RankUp menambah pertanyaan:

```text
apakah sample A punya Y1 lebih besar daripada sample B?
```

Jadi model belajar nilai dan urutan sekaligus.

### Tiga Loss Dalam RankUp

```python
loss = loss_reg + W_RANK * loss_rank_lab + W_UNLAB * loss_rank_unlab
```

Maknanya:

- `loss_reg`: prediksi nilai `Y1` dari data labeled.
- `loss_rank_lab`: belajar ranking dari data labeled.
- `loss_rank_unlab`: belajar konsistensi ranking dari data unlabeled.

### Alur Training

Dalam setiap epoch, `unlab_loader` menjadi loop utama agar semua batch unlabeled dipakai.

```text
ambil 1 batch unlabeled: x_unlab
ambil 1 batch labeled: x_lab, y_lab
jika labeled habis, ulang labeled loader dari awal
```

Lalu training berjalan seperti ini:

```text
1. x_lab, y_lab -> prediksi Y1 -> loss_reg
2. x_lab, y_lab -> ranking pairwise asli -> loss_rank_lab
3. x_unlab diberi weak noise -> model membuat pseudo-label ranking
4. pseudo-label ranking dipakai hanya jika confidence `>= P_CUTOFF`
5. x_unlab diberi strong noise -> model dilatih mengikuti pseudo-label ranking yang lolos threshold
6. semua loss digabung -> backward -> update model
```

### Pseudo-label di RankUp

Pseudo-label di notebook ini bukan nilai `Y1` langsung. Pseudo-label yang dibuat adalah ranking:

```text
sample A lebih besar dari sample B atau tidak?
```

Untuk target ranking, notebook mengikuti ide paper:

```text
A > B  -> target 1
A < B  -> target 0
A == B -> target 0.5
```

Jadi pair yang setara tidak dipaksa menjadi class 0.

### Kenapa Data Unlabeled Bisa Dipakai?

Karena walaupun `x_unlab` tidak punya label asli, model bisa membuat prediksi ranking sementara dari versi weak noise. Jika confidence prediksi weak `>= P_CUTOFF`, prediksi itu dipakai sebagai pseudo-label untuk melatih versi strong noise agar konsisten.

Ini bukan memberi label permanen ke data unlabeled. Pseudo-label dibuat ulang selama training.

In [ ]:
def train_rankup():
    model = RankUpMLP(input_dim=x_lab.shape[1]).to(DEVICE)
    reg_loss_fn = nn.L1Loss()
    rank_loss_fn = nn.CrossEntropyLoss()
    rank_loss_each_fn = nn.CrossEntropyLoss(reduction="none")
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    history = []

    for epoch in range(EPOCHS_RANKUP):
        model.train()
        total_loss = 0
        total_loss_reg = 0
        total_loss_rank_lab = 0
        total_loss_rank_unlab = 0
        total_conf_pair_ratio = 0

        lab_iter = iter(lab_loader)

        for (x_unlab_batch,) in unlab_loader:
            try:
                x_lab_batch, y_lab_batch = next(lab_iter)
            except StopIteration:
                lab_iter = iter(lab_loader)
                x_lab_batch, y_lab_batch = next(lab_iter)

            x_lab_batch = x_lab_batch.to(DEVICE)
            y_lab_batch = y_lab_batch.to(DEVICE)
            x_unlab_batch = x_unlab_batch.to(DEVICE)

            # 1. Loss regresi biasa dari labeled data.
            pred_lab, rank_lab = model(x_lab_batch)
            loss_reg = reg_loss_fn(pred_lab, y_lab_batch)

            # 2. Loss ranking supervised dari labeled data.
            logits_lab = pair_rank_logits(rank_lab)
            targets_lab = pairwise_rank_target(y_lab_batch)
            loss_rank_lab = rank_loss_fn(logits_lab, targets_lab)

            # 3. FixMatch loss pada ARC untuk unlabeled data.
            x_unlab_weak = x_unlab_batch + NOISE_WEAK * torch.randn_like(x_unlab_batch)
            x_unlab_strong = x_unlab_batch + NOISE_STRONG * torch.randn_like(x_unlab_batch)

            with torch.no_grad():
                _, rank_weak = model(x_unlab_weak)
                logits_weak = pair_rank_logits(rank_weak)
                prob_weak = torch.softmax(logits_weak, dim=1)
                conf_weak, pseudo_rank = prob_weak.max(dim=1)
                conf_mask = (conf_weak >= P_CUTOFF).float()

            _, rank_strong = model(x_unlab_strong)
            logits_strong = pair_rank_logits(rank_strong)
            loss_rank_unlab_each = rank_loss_each_fn(logits_strong, pseudo_rank)
            loss_rank_unlab = (loss_rank_unlab_each * conf_mask).sum() / conf_mask.sum().clamp(min=1.0)

            loss = loss_reg + W_RANK * loss_rank_lab + W_UNLAB * loss_rank_unlab

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            total_loss_reg += loss_reg.item()
            total_loss_rank_lab += loss_rank_lab.item()
            total_loss_rank_unlab += loss_rank_unlab.item()
            total_conf_pair_ratio += conf_mask.mean().item()

        num_batches = len(unlab_loader)
        history.append({
            "total": total_loss / num_batches,
            "regression": total_loss_reg / num_batches,
            "rank_lab": total_loss_rank_lab / num_batches,
            "rank_unlab": total_loss_rank_unlab / num_batches,
            "conf_pair_ratio": total_conf_pair_ratio / num_batches,
        })

    return model, history


rankup_model, rankup_history = train_rankup()
rankup_metrics = evaluate(rankup_model)
print_metrics("RankUp minimal", rankup_metrics)


## 10. Fully-Supervised Baseline

Model ini memakai semua label pada train set:

```text
x_train + y_train
```

Ini bukan semi-supervised. Tujuannya hanya sebagai **upper bound**: kira-kira seberapa bagus hasil jika semua label training diketahui.

Perbandingan utama semi-supervised tetap:

```text
Supervised vs RankUp
```

Fully-Supervised hanya pembanding tambahan.

In [ ]:
def train_fully_supervised():
    model = RankUpMLP(input_dim=x_lab.shape[1]).to(DEVICE)
    loss_fn = nn.L1Loss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    history = []

    for epoch in range(EPOCHS_FULL):
        model.train()
        total_loss = 0

        for x_train_batch, y_train_batch in train_loader:
            x_train_batch = x_train_batch.to(DEVICE)
            y_train_batch = y_train_batch.to(DEVICE)

            pred_batch, _ = model(x_train_batch)
            loss = loss_fn(pred_batch, y_train_batch)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        history.append(total_loss / len(train_loader))

    return model, history


fully_model, fully_history = train_fully_supervised()
fully_metrics = evaluate(fully_model)
print_metrics("Fully-Supervised", fully_metrics)


## 11. Perbandingan Singkat

Tabel ini membandingkan tiga model pada test set yang sama.

- **Supervised**: label sedikit, tanpa unlabeled.
- **RankUp minimal**: label sedikit + unlabeled.
- **Fully-Supervised**: semua label train, sebagai upper bound.

Metric:

- `MAE`: lebih kecil lebih baik.
- `R2`: lebih besar lebih baik.
- `Spearman`: lebih besar lebih baik, karena mengukur kualitas ranking.

In [ ]:
summary = pd.DataFrame(
    [supervised_metrics, rankup_metrics, fully_metrics],
    index=["Supervised", "RankUp minimal", "Fully-Supervised"],
)
summary

## 12. Visualisasi Training, Metrik, dan Prediksi

Visualisasi dibuat untuk melihat:

1. perbandingan metrik antar model,
2. kurva loss selama training,
3. komponen loss pada RankUp,
4. prediksi vs target asli,
5. distribusi residual error.

Untuk `MAE`, lebih kecil lebih baik. Untuk `R2` dan `Spearman`, lebih besar lebih baik.

In [ ]:
metrics_df = pd.DataFrame([
    {"Model": "Supervised", **supervised_metrics},
    {"Model": "RankUp", **rankup_metrics},
    {"Model": "Fully-Supervised", **fully_metrics},
])

metrics_long = metrics_df.melt(
    id_vars="Model",
    var_name="Metric",
    value_name="Value",
)

plt.figure(figsize=(9, 4))
sns.barplot(data=metrics_long, x="Metric", y="Value", hue="Model")
plt.title("Perbandingan Metrik Evaluasi")
plt.xlabel("Metric")
plt.ylabel("Value")
plt.legend(title="Model")
plt.show()

metrics_df

In [ ]:
loss_supervised_df = pd.DataFrame({
    "Epoch": range(1, len(supervised_history) + 1),
    "Loss": supervised_history,
    "Model": "Supervised",
})

loss_rankup_df = pd.DataFrame({
    "Epoch": range(1, len(rankup_history) + 1),
    "Loss": [item["total"] for item in rankup_history],
    "Model": "RankUp",
})

loss_full_df = pd.DataFrame({
    "Epoch": range(1, len(fully_history) + 1),
    "Loss": fully_history,
    "Model": "Fully-Supervised",
})

loss_df = pd.concat([loss_supervised_df, loss_rankup_df, loss_full_df], ignore_index=True)

plt.figure(figsize=(9, 4))
sns.lineplot(data=loss_df, x="Epoch", y="Loss", hue="Model")
plt.title("Training Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.show()

rankup_loss_df = pd.DataFrame(rankup_history)
rankup_loss_df["Epoch"] = range(1, len(rankup_loss_df) + 1)
rankup_loss_long = rankup_loss_df.melt(
    id_vars="Epoch",
    var_name="Loss Type",
    value_name="Loss",
)

plt.figure(figsize=(9, 4))
sns.lineplot(data=rankup_loss_long, x="Epoch", y="Loss", hue="Loss Type")
plt.title("Breakdown Loss RankUp")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.show()

In [ ]:
pred_supervised, target_real = predict_test(supervised_model)
pred_rankup, _ = predict_test(rankup_model)
pred_full, _ = predict_test(fully_model)

pred_df = pd.DataFrame({
    "Target": np.concatenate([target_real, target_real, target_real]),
    "Prediction": np.concatenate([pred_supervised, pred_rankup, pred_full]),
    "Model": ["Supervised"] * len(target_real) + ["RankUp"] * len(target_real) + ["Fully-Supervised"] * len(target_real),
})

min_value = min(pred_df["Target"].min(), pred_df["Prediction"].min())
max_value = max(pred_df["Target"].max(), pred_df["Prediction"].max())

plt.figure(figsize=(6, 6))
sns.scatterplot(data=pred_df, x="Target", y="Prediction", hue="Model", alpha=0.75)
plt.plot([min_value, max_value], [min_value, max_value], "--", color="black", label="Ideal")
plt.title("Prediksi vs Target Asli")
plt.xlabel("Target asli")
plt.ylabel("Prediksi")
plt.legend()
plt.show()

residual_df = pred_df.copy()
residual_df["Residual"] = residual_df["Prediction"] - residual_df["Target"]

plt.figure(figsize=(9, 4))
sns.histplot(data=residual_df, x="Residual", hue="Model", kde=True, bins=20, alpha=0.45)
plt.axvline(0, color="black", linestyle="--")
plt.title("Distribusi Residual Error")
plt.xlabel("Prediction - Target")
plt.ylabel("Jumlah sample")
plt.show()


## 13. Visualisasi t-SNE Feature Representation

Visualisasi ini mirip ide pada paper: kita ambil representasi hidden dari `encoder`, lalu diproyeksikan ke 2D memakai t-SNE.

Tujuannya melihat apakah representasi fitur dari model membentuk pola yang lebih terurut terhadap nilai target `Y1`.


In [ ]:
def get_encoder_features(model, x_tensor):
    model.eval()

    with torch.no_grad():
        z = model.encoder(x_tensor.to(DEVICE))

    return z.cpu().numpy()


def make_tsne_embedding(model, model_name):
    z = get_encoder_features(model, x_test_tensor)
    perplexity = min(30, len(z) - 1)

    tsne = TSNE(
        n_components=2,
        perplexity=perplexity,
        init="pca",
        learning_rate="auto",
        random_state=SEED,
    )
    z_2d = tsne.fit_transform(z)

    return pd.DataFrame({
        "TSNE-1": z_2d[:, 0],
        "TSNE-2": z_2d[:, 1],
        "Y1": y_test_raw.reshape(-1),
        "Model": model_name,
    })


tsne_dfs = [
    make_tsne_embedding(supervised_model, "Supervised"),
    make_tsne_embedding(rankup_model, "RankUp"),
    make_tsne_embedding(fully_model, "Fully-Supervised"),
]

fig, axes = plt.subplots(1, 3, figsize=(15, 4), constrained_layout=True)
vmin = min(tsne_df["Y1"].min() for tsne_df in tsne_dfs)
vmax = max(tsne_df["Y1"].max() for tsne_df in tsne_dfs)

for ax, tsne_df in zip(axes, tsne_dfs):
    scatter = ax.scatter(
        tsne_df["TSNE-1"],
        tsne_df["TSNE-2"],
        c=tsne_df["Y1"],
        cmap="viridis",
        vmin=vmin,
        vmax=vmax,
        s=28,
        alpha=0.85,
    )
    ax.set_title(tsne_df["Model"].iloc[0])
    ax.set_xlabel("t-SNE 1")
    ax.set_ylabel("t-SNE 2")

fig.colorbar(scatter, ax=axes, label="Heating Load (Y1)")
plt.suptitle("t-SNE Feature Representation dari Encoder")
plt.show()


## 14. Eksperimen yang Bisa Dicoba

Ubah satu parameter setiap eksperimen agar efeknya mudah dibaca.

### `N_LABEL`

Mengatur jumlah label yang boleh dilihat oleh Supervised dan RankUp.

Contoh:

```python
N_LABEL = 20
N_LABEL = 80
N_LABEL = 200
```

Jika label terlalu sedikit, Auxiliary Ranking Classifier (ARC) bisa belum stabil. Jika label lebih banyak, ranking pairwise biasanya lebih bagus.

### `W_RANK`

Bobot untuk ranking loss dari labeled data.

```python
W_RANK = 0.01
W_RANK = 0.05
W_RANK = 0.1
```

Kalau terlalu besar, model bisa terlalu fokus ke ranking dan mengganggu regresi.

### `W_UNLAB`

Bobot untuk consistency loss dari unlabeled data.

```python
W_UNLAB = 0.0
W_UNLAB = 0.05
W_UNLAB = 0.1
```

`0.0` berarti RankUp hanya memakai Auxiliary Ranking Classifier (ARC) supervised, tanpa FixMatch loss dari unlabeled data.

### `P_CUTOFF`

Threshold confidence dari weak ARC. Pair unlabeled hanya dipakai jika confidence ranking-nya cukup tinggi.

```python
P_CUTOFF = 0.8
P_CUTOFF = 0.9
P_CUTOFF = 0.95
```

Kalau terlalu tinggi, sedikit pair yang dipakai. Kalau terlalu rendah, pseudo-label ranking yang salah bisa ikut masuk.

### `EPOCHS_RANKUP`

Mengatur lama training RankUp.

```python
EPOCHS_RANKUP = 50
EPOCHS_RANKUP = 100
EPOCHS_RANKUP = 200
```

Jika epoch terlalu kecil model belum belajar. Jika terlalu besar bisa overfit.